### Setup

Imports

In [ ]:
import os
import re
import sys
import importlib
import math
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union
from netCDF4 import Dataset, num2date
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xarray as xr
from plotly.subplots import make_subplots

Import helper modules from /Tools/helpers

In [ ]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

# Read the metadata table
from tools.database_lookup import (
    get_instrument_context,
    resolve_pressure_source_by_id,
)

# Generic helpers
from tools.helpers import clean_label, is_missing, plot_data_by_qc, resolve_good_data_window, safe_tag, save_plotly_figure

# Plotting helpers
from tools.helpers.plot_qa_qc import _to_py_dt

# Pressure sensor comparison helper 
from tools.helpers import plot_pressure_comparison

# Apply Atmospheric Pressure Offset
from tools.helpers import apply_atmospheric_pressure_offset


Definitions

In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 270
# 202603:
#           SBE37   SBE26/RBR
#   BASJAS  266     265
#   BASS3A  270     269
#   BASS3B  273     272

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    metadata_csv="/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/reference_mooring_proc_info/satellite_altimetry_moorings_metadata.csv",
    print_details=True,
)


Read and create ds, df

In [ ]:
# Read cnv
from tools.parsers.read_sbe37 import read_sbe37_cnv

df = read_sbe37_cnv(cfg["input_file"], verbose=True)

##### Optional: Add/Remove atmospheric pressure to get absolute pressure
Use this when the atmospheric pressure offset has been applied in the instrument setup ONLY.
In this instance (rec_202603 BASJAS SBE37_22467) atmospheric pressure of 10.1325 dbar was applied during sensor's onboard processing, therefore the "PRES" variable is actually relative pressure and not absolute.


In [ ]:
# Sanity Check
print("PRES min/max:", float(np.nanmin(ds["PRES"].values)), float(np.nanmax(ds["PRES"].values)))

In [ ]:
# Atmospheric pressure adjustment (set after inspecting prior plots/QC if needed)
apply_atm_pressure = True      # toggle on/off
atm_offset_dbar = 10.1325      # change this value in-place when needed
atm_mode = "add"               # "add" for relative->absolute in your described case

if apply_atm_pressure:
    ds = apply_atmospheric_pressure_offset(
        ds,
        offset_dbar=atm_offset_dbar,
        pres_var="PRES",
        mode=atm_mode,
        in_place=True,
    )
else:
    print("Atmospheric pressure adjustment skipped.")

In [ ]:
# Sanity Check
print("PRES min/max:", float(np.nanmin(ds["PRES"].values)), float(np.nanmax(ds["PRES"].values)))

In [ ]:
# sync corrected PRES from ds back to df for pressure comparison plotting
df["Pressure [db]"] = ds["PRES"].values


ASHLEY * Check this!!!! ^^^^^

In [ ]:
fig_press, comp_label_used, comp_file_used = plot_pressure_comparison(
        df=df,
        database=database,
        pressure_inst_deploy_id=pressure_inst_deploy_id,
        pressure_file=pressure_file,
        comparison_label=comparison_label,
        x_start=x_start,
        x_end=x_end,
        y_zoom_to_good=False,
    )
fig_press.show()

### Save to IMOS-compliant NetCDF
